In [1]:
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd
import sklearn

project_dir = Path.cwd().parent
raw_dir = project_dir / "data" / "raw"
artifacts_dir = Path.cwd() / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

print("Project:", project_dir)
print("Raw data exists:", raw_dir.exists())
print("scikit-learn:", sklearn.__version__)
print("CatBoost installed:", importlib.util.find_spec("catboost") is not None)

Project: c:\Dev\Projects\planet-hunter
Raw data exists: True
scikit-learn: 1.9.1
CatBoost installed: False


In [3]:
import catboost
print("CatBoost version:", catboost.__version__)

CatBoost version: 1.2.10


In [4]:
from pathlib import Path

project_dir = Path.cwd().parent

for folder in [
    project_dir / "data" / "processed",
    project_dir / "notebooks" / "artifacts",
]:
    print(f"\n{folder}")
    if folder.exists():
        for path in folder.iterdir():
            print(" ", path.name)
    else:
        print("  Folder does not exist")


c:\Dev\Projects\planet-hunter\data\processed

c:\Dev\Projects\planet-hunter\notebooks\artifacts
  submission.csv
  submission_2.csv
  submission_3_starspots.csv
  submission_4_blend.csv


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Raw files:")
for path in RAW_DIR.iterdir():
    print(path.name)

Raw files:
sample_submission.csv
test_cadences.csv
test_centroid.npy
test_flux.npy
test_stars.csv
train_cadences.csv
train_centroid.npy
train_flux.npy
train_stars.csv


In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Load training data
train_flux = np.load(RAW_DIR / "train_flux.npy")
train_centroid = np.load(RAW_DIR / "train_centroid.npy")
train_stars = pd.read_csv(RAW_DIR / "train_stars.csv")
train_cadences = pd.read_csv(RAW_DIR / "train_cadences.csv")

# Load test data
test_flux = np.load(RAW_DIR / "test_flux.npy")
test_centroid = np.load(RAW_DIR / "test_centroid.npy")
test_stars = pd.read_csv(RAW_DIR / "test_stars.csv")
test_cadences = pd.read_csv(RAW_DIR / "test_cadences.csv")

# Confirm label column
print("Train stars columns:", train_stars.columns.tolist())
print("Train flux shape:", train_flux.shape)
print("Test flux shape:", test_flux.shape)
print("Train centroid shape:", train_centroid.shape)
print("Test centroid shape:", test_centroid.shape)
print("Train cadence columns:", train_cadences.columns.tolist())

Train stars columns: ['id', 'tmag', 'teff', 'logg', 'radius', 'contamination', 'camera', 'ccd', 'label']
Train flux shape: (18000, 1315)
Test flux shape: (7000, 1315)
Train centroid shape: (18000, 2, 1315)
Test centroid shape: (7000, 2, 1315)
Train cadence columns: ['cadence', 'time_btjd', 'quality']


In [7]:
# Quality mask: retain cadences where quality == 0
quality_mask = (
    train_cadences["quality"].to_numpy() == 0
)
test_quality_mask = (
    test_cadences["quality"].to_numpy() == 0
)

train_flux_q = train_flux[:, quality_mask]
test_flux_q = test_flux[:, test_quality_mask]

train_centroid_q = train_centroid[:, :, quality_mask]
test_centroid_q = test_centroid[:, :, test_quality_mask]

y = train_stars["label"].copy()

print("Train quality-filtered flux:", train_flux_q.shape)
print("Test quality-filtered flux:", test_flux_q.shape)
print("Train quality-filtered centroid:", train_centroid_q.shape)
print("Test quality-filtered centroid:", test_centroid_q.shape)
print("Training labels:", y.shape)
print("\nClass distribution:")
print(y.value_counts())

Train quality-filtered flux: (18000, 1221)
Test quality-filtered flux: (7000, 1219)
Train quality-filtered centroid: (18000, 2, 1221)
Test quality-filtered centroid: (7000, 2, 1219)
Training labels: (18000,)

Class distribution:
label
quiet_star           3624
starspots            2882
eclipsing_binary     2247
planet               2145
pulsating_star       1973
flare_star           1804
instrument_glitch    1721
background_binary    1604
Name: count, dtype: int64


In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.features.centroid import extract_centroid_features
from src.features.periodic import extract_periodic_features
from src.features.artifacts import extract_artifact_features
from src.features.dip import dip_features

print("Feature modules imported successfully.")

Feature modules imported successfully.


In [11]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis


def extract_base_features(flux, stars, batch=2000):
    output = []

    for start in range(0, len(flux), batch):
        values = np.asarray(
            flux[start:start + batch], dtype=np.float32
        )

        median = np.nanmedian(values, axis=1, keepdims=True)
        normalized = values / np.where(
            np.abs(median) < 1e-8, np.nan, median
        )

        stats = np.column_stack([
            np.nanstd(normalized, axis=1),
            np.nanmedian(
                np.abs(
                    normalized
                    - np.nanmedian(normalized, axis=1, keepdims=True)
                ),
                axis=1,
            ),
            np.nanmin(normalized, axis=1),
            np.nanmax(normalized, axis=1),
            skew(normalized, axis=1, nan_policy="omit"),
            kurtosis(normalized, axis=1, nan_policy="omit"),
            *[
                np.nanpercentile(normalized, q, axis=1)
                for q in [1, 5, 25, 75, 95, 99]
            ],
        ])
        output.append(stats)

    flux_features = np.vstack(output)
    stellar_features = stars[
        ["tmag", "teff", "logg", "radius", "contamination"]
    ].to_numpy(dtype=np.float32)

    names = [
        "std", "mad", "min", "max", "skew", "kurtosis",
        "p01", "p05", "p25", "p75", "p95", "p99",
    ]

    return pd.DataFrame(
        np.column_stack([flux_features, stellar_features]),
        columns=names + [
            "tmag", "teff", "logg", "radius", "contamination"
        ],
    )


def extract_temporal_features(flux):
    flux = np.asarray(flux, dtype=np.float32)

    median = np.nanmedian(flux, axis=1, keepdims=True)
    normalized = flux / np.where(
        np.abs(median) < 1e-8, np.nan, median
    )

    differences = np.diff(normalized, axis=1)
    diff_std = np.nanstd(differences, axis=1)
    diff_abs_median = np.nanmedian(np.abs(differences), axis=1)

    x = np.arange(normalized.shape[1], dtype=np.float32)
    x_centered = x - x.mean()

    y_centered = normalized - np.nanmean(
        normalized, axis=1, keepdims=True
    )
    trend_slope = np.nansum(
        y_centered * x_centered, axis=1
    ) / max(np.sum(x_centered ** 2), 1e-8)

    return pd.DataFrame({
        "trend_slope": trend_slope,
        "diff_abs_median": diff_abs_median,
        "diff_std": diff_std,
    })


print("Base and temporal feature functions defined.")

Base and temporal feature functions defined.


In [12]:
# Base features
X_base = extract_base_features(train_flux_q, train_stars)
X_test_base = extract_base_features(test_flux_q, test_stars)

# Temporal features
X_temporal_all = extract_temporal_features(train_flux_q)
X_test_temporal_all = extract_temporal_features(test_flux_q)

temporal_cols = ["trend_slope", "diff_abs_median", "diff_std"]

X_temporal = X_temporal_all[temporal_cols]
X_test_temporal = X_test_temporal_all[temporal_cols]

# Centroid features
X_centroid = extract_centroid_features(train_centroid_q)
X_test_centroid = extract_centroid_features(test_centroid_q)

# Periodic features
X_periodic = extract_periodic_features(train_flux_q)
X_test_periodic = extract_periodic_features(test_flux_q)

# Artifact features
X_artifact = extract_artifact_features(train_flux_q)
X_test_artifact = extract_artifact_features(test_flux_q)

# Dip features
X_dip = dip_features(train_flux_q)
X_test_dip = dip_features(test_flux_q)

# Combine
X_quality_dip = pd.concat(
    [X_base, X_temporal, X_centroid, X_periodic, X_artifact, X_dip],
    axis=1,
)

X_test_quality_dip = pd.concat(
    [
        X_test_base, X_test_temporal, X_test_centroid,
        X_test_periodic, X_test_artifact, X_test_dip,
    ],
    axis=1,
)

X_test_quality_dip = X_test_quality_dip[X_quality_dip.columns]

print("Train:", X_quality_dip.shape)
print("Test:", X_test_quality_dip.shape)
print("Same columns:",
      X_quality_dip.columns.equals(X_test_quality_dip.columns))
print("Train finite:",
      np.isfinite(X_quality_dip.to_numpy()).all())
print("Test finite:",
      np.isfinite(X_test_quality_dip.to_numpy()).all())

Train: (18000, 40)
Test: (7000, 40)
Same columns: True
Train finite: True
Test finite: True


In [13]:
print("Train:", X_quality_dip.shape)
print("Test:", X_test_quality_dip.shape)
print("Same columns:",
      X_quality_dip.columns.equals(X_test_quality_dip.columns))
print("Train finite:",
      np.isfinite(X_quality_dip.to_numpy()).all())
print("Test finite:",
      np.isfinite(X_test_quality_dip.to_numpy()).all())

print("\nFeature columns:")
print(X_quality_dip.columns.tolist())

Train: (18000, 40)
Test: (7000, 40)
Same columns: True
Train finite: True
Test finite: True

Feature columns:
['std', 'mad', 'min', 'max', 'skew', 'kurtosis', 'p01', 'p05', 'p25', 'p75', 'p95', 'p99', 'tmag', 'teff', 'logg', 'radius', 'contamination', 'trend_slope', 'diff_abs_median', 'diff_std', 'centroid_x_std', 'centroid_x_mad', 'centroid_x_diff_median', 'centroid_x_dev_p95', 'centroid_y_std', 'centroid_y_mad', 'centroid_y_diff_median', 'centroid_y_dev_p95', 'autocorr_lag_5', 'autocorr_lag_10', 'autocorr_lag_20', 'autocorr_lag_50', 'autocorr_lag_100', 'extreme_outlier_fraction', 'max_consecutive_jump', 'large_jump_fraction', 'dip_fraction', 'deepest_dip', 'dip_asymmetry', 'dip_concentration']


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from catboost import CatBoostClassifier
import time

X = X_quality_dip
y = train_stars["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

cat_model = CatBoostClassifier(
    iterations=1000,
    depth=7,
    learning_rate=0.05,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
    allow_writing_files=False,
    thread_count=-1,
)

start = time.time()

cat_model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    verbose=100,
)

val_pred = cat_model.predict(X_val).ravel()

cat_f1 = f1_score(y_val, val_pred, average="macro")

print(f"\nCatBoost validation macro-F1: {cat_f1:.4f}")
print(f"Training time: {(time.time() - start) / 60:.1f} minutes")
print(classification_report(y_val, val_pred, zero_division=0))

0:	learn: 2.0353901	test: 2.0361005	best: 2.0361005 (0)	total: 328ms	remaining: 5m 27s
100:	learn: 1.3763614	test: 1.4574407	best: 1.4574407 (100)	total: 14.7s	remaining: 2m 11s
200:	learn: 1.2443596	test: 1.3955460	best: 1.3955460 (200)	total: 29s	remaining: 1m 55s
300:	learn: 1.1476712	test: 1.3652338	best: 1.3652338 (300)	total: 43.7s	remaining: 1m 41s
400:	learn: 1.0690648	test: 1.3474540	best: 1.3474540 (400)	total: 58.7s	remaining: 1m 27s
500:	learn: 1.0039685	test: 1.3376835	best: 1.3376835 (500)	total: 1m 13s	remaining: 1m 13s
600:	learn: 0.9423698	test: 1.3318053	best: 1.3318053 (600)	total: 1m 28s	remaining: 58.9s
700:	learn: 0.8888553	test: 1.3266491	best: 1.3265295 (697)	total: 1m 43s	remaining: 43.9s
800:	learn: 0.8388394	test: 1.3233428	best: 1.3232668 (797)	total: 1m 57s	remaining: 29.1s
900:	learn: 0.7911065	test: 1.3214930	best: 1.3214930 (900)	total: 2m 11s	remaining: 14.4s
999:	learn: 0.7475952	test: 1.3191391	best: 1.3191391 (999)	total: 2m 25s	remaining: 0us

bestT

In [15]:
cat_model_1500 = CatBoostClassifier(
    iterations=1500,
    depth=7,
    learning_rate=0.05,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
    allow_writing_files=False,
    thread_count=-1,
)

cat_model_1500.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=150,
    verbose=100,
)

val_pred_1500 = cat_model_1500.predict(X_val).ravel()

cat_f1_1500 = f1_score(
    y_val, val_pred_1500, average="macro"
)

print(f"CatBoost 1500-iteration macro-F1: {cat_f1_1500:.4f}")
print("Best iteration:", cat_model_1500.best_iteration_)
print("Best validation loss:", cat_model_1500.best_score_["validation"]["MultiClass"])

0:	learn: 2.0353901	test: 2.0361005	best: 2.0361005 (0)	total: 162ms	remaining: 4m 3s
100:	learn: 1.3763614	test: 1.4574407	best: 1.4574407 (100)	total: 14.7s	remaining: 3m 24s
200:	learn: 1.2443596	test: 1.3955460	best: 1.3955460 (200)	total: 28.8s	remaining: 3m 6s
300:	learn: 1.1476712	test: 1.3652338	best: 1.3652338 (300)	total: 43.5s	remaining: 2m 53s
400:	learn: 1.0690648	test: 1.3474540	best: 1.3474540 (400)	total: 58.1s	remaining: 2m 39s
500:	learn: 1.0039685	test: 1.3376835	best: 1.3376835 (500)	total: 1m 30s	remaining: 3m
600:	learn: 0.9423698	test: 1.3318053	best: 1.3318053 (600)	total: 2m 6s	remaining: 3m 8s
700:	learn: 0.8888553	test: 1.3266491	best: 1.3265295 (697)	total: 2m 37s	remaining: 2m 59s
800:	learn: 0.8388394	test: 1.3233428	best: 1.3232668 (797)	total: 2m 55s	remaining: 2m 33s
900:	learn: 0.7911065	test: 1.3214930	best: 1.3214930 (900)	total: 3m 10s	remaining: 2m 6s
1000:	learn: 0.7471842	test: 1.3193677	best: 1.3191391 (999)	total: 3m 25s	remaining: 1m 42s
1100:

In [16]:
cat_model_d6 = CatBoostClassifier(
    iterations=1500,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
    allow_writing_files=False,
    thread_count=-1,
)

cat_model_d6.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=150,
    verbose=100,
)

val_pred_d6 = cat_model_d6.predict(X_val).ravel()

print(
    "Depth 6 macro-F1:",
    f1_score(y_val, val_pred_d6, average="macro")
)

0:	learn: 2.0379077	test: 2.0385283	best: 2.0385283 (0)	total: 93.2ms	remaining: 2m 19s
100:	learn: 1.4142758	test: 1.4668064	best: 1.4668064 (100)	total: 11.4s	remaining: 2m 37s
200:	learn: 1.3029763	test: 1.4036315	best: 1.4036315 (200)	total: 23.9s	remaining: 2m 34s
300:	learn: 1.2254881	test: 1.3705036	best: 1.3705036 (300)	total: 36.2s	remaining: 2m 24s
400:	learn: 1.1608246	test: 1.3506815	best: 1.3506815 (400)	total: 42.7s	remaining: 1m 57s
500:	learn: 1.1076211	test: 1.3383206	best: 1.3383206 (500)	total: 48.8s	remaining: 1m 37s
600:	learn: 1.0589028	test: 1.3275338	best: 1.3274978 (599)	total: 55.6s	remaining: 1m 23s
700:	learn: 1.0154996	test: 1.3211308	best: 1.3210994 (699)	total: 1m 1s	remaining: 1m 10s
800:	learn: 0.9759731	test: 1.3156464	best: 1.3156257 (798)	total: 1m 8s	remaining: 59.6s
900:	learn: 0.9377481	test: 1.3119057	best: 1.3119057 (900)	total: 1m 14s	remaining: 49.7s
1000:	learn: 0.9027059	test: 1.3084022	best: 1.3084022 (1000)	total: 1m 20s	remaining: 40.3s
1

In [17]:
cat_model_d6_2000 = CatBoostClassifier(
    iterations=2000,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=200,
    allow_writing_files=False,
    thread_count=-1,
)

cat_model_d6_2000.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=200,
    verbose=200,
)

val_pred = cat_model_d6_2000.predict(X_val).ravel()

print("Macro-F1:", f1_score(y_val, val_pred, average="macro"))
print("Best iteration:", cat_model_d6_2000.best_iteration_)

0:	learn: 2.0379077	test: 2.0385283	best: 2.0385283 (0)	total: 73.8ms	remaining: 2m 27s
200:	learn: 1.3029763	test: 1.4036315	best: 1.4036315 (200)	total: 12.1s	remaining: 1m 48s
400:	learn: 1.1608246	test: 1.3506815	best: 1.3506815 (400)	total: 23.9s	remaining: 1m 35s
600:	learn: 1.0589028	test: 1.3275338	best: 1.3274978 (599)	total: 36.6s	remaining: 1m 25s
800:	learn: 0.9759731	test: 1.3156464	best: 1.3156257 (798)	total: 49.2s	remaining: 1m 13s
1000:	learn: 0.9027059	test: 1.3084022	best: 1.3084022 (1000)	total: 1m 1s	remaining: 1m
1200:	learn: 0.8382886	test: 1.3050725	best: 1.3050336 (1196)	total: 1m 12s	remaining: 48.5s
1400:	learn: 0.7797658	test: 1.3032662	best: 1.3032662 (1400)	total: 1m 24s	remaining: 36.3s
1600:	learn: 0.7241906	test: 1.3029701	best: 1.3019369 (1522)	total: 1m 36s	remaining: 24.2s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 1.301936882
bestIteration = 1522

Shrink model to first 1523 iterations.
Macro-F1: 0.5175469806836518
Best iterat

In [19]:
cat_model_d5_2000 = CatBoostClassifier(
    iterations=2000,
    depth=5,
    learning_rate=0.05,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=200,
    allow_writing_files=False,
    thread_count=-1,
)

cat_model_d5_2000.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=200,
    verbose=200,
)

val_pred = cat_model_d5_2000.predict(X_val).ravel()

print("Macro-F1:", f1_score(y_val, val_pred, average="macro"))
print("Best iteration:", cat_model_d5_2000.best_iteration_)

0:	learn: 2.0380999	test: 2.0384443	best: 2.0384443 (0)	total: 54.7ms	remaining: 1m 49s
200:	learn: 1.3550693	test: 1.4213467	best: 1.4213467 (200)	total: 7.42s	remaining: 1m 6s
400:	learn: 1.2323064	test: 1.3581763	best: 1.3581763 (400)	total: 15.1s	remaining: 1m
600:	learn: 1.1524370	test: 1.3317102	best: 1.3317102 (600)	total: 26s	remaining: 1m
800:	learn: 1.0912983	test: 1.3183022	best: 1.3182574 (796)	total: 39.2s	remaining: 58.6s
1000:	learn: 1.0372783	test: 1.3099960	best: 1.3099352 (999)	total: 52.6s	remaining: 52.5s
1200:	learn: 0.9890912	test: 1.3044507	best: 1.3044507 (1200)	total: 1m 1s	remaining: 41.2s
1400:	learn: 0.9433291	test: 1.2981322	best: 1.2981135 (1396)	total: 1m 10s	remaining: 29.9s
1600:	learn: 0.9012546	test: 1.2948005	best: 1.2947794 (1596)	total: 1m 17s	remaining: 19.2s
1800:	learn: 0.8602412	test: 1.2917852	best: 1.2916302 (1787)	total: 1m 24s	remaining: 9.39s
1999:	learn: 0.8225189	test: 1.2897761	best: 1.2896812 (1980)	total: 1m 32s	remaining: 0us

bestTe

In [26]:
cat_model_d5_2500 = CatBoostClassifier(
    iterations=2500,
    depth=5,
    learning_rate=0.05,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=250,
    allow_writing_files=False,
    thread_count=-1,
)

cat_model_d5_2500.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=200,
    verbose=250,
)

val_pred_d5 = cat_model_d5_2500.predict(X_val).ravel()

print("Macro-F1:", f1_score(y_val, val_pred_d5, average="macro"))
print("Best iteration:", cat_model_d5_2500.best_iteration_)

0:	learn: 2.0380999	test: 2.0384443	best: 2.0384443 (0)	total: 54.6ms	remaining: 2m 16s
250:	learn: 1.3192096	test: 1.4002564	best: 1.4002564 (250)	total: 9.07s	remaining: 1m 21s
500:	learn: 1.1902358	test: 1.3431051	best: 1.3431051 (500)	total: 17.8s	remaining: 1m 10s
750:	learn: 1.1052675	test: 1.3205004	best: 1.3205004 (750)	total: 26.6s	remaining: 1m 2s
1000:	learn: 1.0372783	test: 1.3099960	best: 1.3099352 (999)	total: 35.5s	remaining: 53.1s
1250:	learn: 0.9779269	test: 1.3034529	best: 1.3034529 (1250)	total: 44.2s	remaining: 44.2s
1500:	learn: 0.9215392	test: 1.2964065	best: 1.2964065 (1500)	total: 53.1s	remaining: 35.3s
1750:	learn: 0.8707696	test: 1.2922575	best: 1.2921695 (1743)	total: 1m 1s	remaining: 26.5s
2000:	learn: 0.8224212	test: 1.2897737	best: 1.2896812 (1980)	total: 1m 10s	remaining: 17.7s
2250:	learn: 0.7784414	test: 1.2881975	best: 1.2878920 (2244)	total: 1m 19s	remaining: 8.81s
2499:	learn: 0.7375107	test: 1.2873378	best: 1.2872540 (2487)	total: 1m 28s	remaining: 

In [28]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import f1_score

hgb_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.06,
    class_weight="balanced",
    random_state=0
)

et_model = ExtraTreesClassifier(
    n_estimators=500,
    min_samples_leaf=2,
    max_features=0.9,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

hgb_model.fit(X_train, y_train)
et_model.fit(X_train, y_train)

hgb_val_proba = hgb_model.predict_proba(X_val)
et_val_proba = et_model.predict_proba(X_val)

# Align ExtraTrees probability columns to HGB's class order
et_indices = [list(et_model.classes_).index(c) for c in hgb_model.classes_]
et_val_proba = et_val_proba[:, et_indices]

hgb_et_val_proba = (
    0.75 * hgb_val_proba
    + 0.25 * et_val_proba
)

hgb_et_pred = hgb_model.classes_[
    np.argmax(hgb_et_val_proba, axis=1)
]

print(
    "HGB + ExtraTrees macro-F1:",
    f1_score(y_val, hgb_et_pred, average="macro")
)

HGB + ExtraTrees macro-F1: 0.5338981332635635


In [29]:
from sklearn.metrics import f1_score
import numpy as np

# Existing HGB + ExtraTrees validation probabilities
# hgb_et_val_proba

# CatBoost depth-5, lr=0.05 validation probabilities
cat_val_proba = cat_model_d5_2500.predict_proba(X_val)

# Align probability columns to the same class order first.
# Use the same ordered class list for both models.
for weight in [0.1, 0.2, 0.3, 0.4]:
    blended_proba = (
        (1 - weight) * hgb_et_val_proba
        + weight * cat_val_proba
    )

    blended_pred = cat_model_d5_2500.classes_[
        np.argmax(blended_proba, axis=1)
    ]

    score = f1_score(y_val, blended_pred, average="macro")
    print(f"CatBoost weight={weight:.1f}: macro-F1={score:.4f}")

CatBoost weight=0.1: macro-F1=0.5331
CatBoost weight=0.2: macro-F1=0.5338
CatBoost weight=0.3: macro-F1=0.5354
CatBoost weight=0.4: macro-F1=0.5366


In [30]:
for weight in [0.5, 0.6, 0.7]:
    blended_proba = (
        (1 - weight) * hgb_et_val_proba
        + weight * cat_val_proba
    )

    blended_pred = hgb_model.classes_[
        np.argmax(blended_proba, axis=1)
    ]

    score = f1_score(y_val, blended_pred, average="macro")
    print(f"CatBoost weight={weight:.1f}: macro-F1={score:.4f}")

CatBoost weight=0.5: macro-F1=0.5352
CatBoost weight=0.6: macro-F1=0.5360
CatBoost weight=0.7: macro-F1=0.5340


In [31]:
# All three models must first be fitted on the full training data.

hgb_test_proba = hgb_model.predict_proba(X_test_quality_dip)
et_test_proba = et_model.predict_proba(X_test_quality_dip)
cat_test_proba = cat_model_d5_2500.predict_proba(X_test_quality_dip)

# Align ExtraTrees and CatBoost columns to HGB's class order.
classes = list(hgb_model.classes_)

et_idx = [list(et_model.classes_).index(c) for c in classes]
cat_idx = [list(cat_model_d5_2500.classes_).index(c) for c in classes]

et_test_proba = et_test_proba[:, et_idx]
cat_test_proba = cat_test_proba[:, cat_idx]

hgb_et_test_proba = (
    0.75 * hgb_test_proba + 0.25 * et_test_proba
)

final_proba = (
    0.60 * hgb_et_test_proba + 0.40 * cat_test_proba
)

final_pred = np.array(classes)[np.argmax(final_proba, axis=1)]

In [33]:
import numpy as np
import pandas as pd

classes = list(hgb_model.classes_)

et_idx = [list(et_model.classes_).index(c) for c in classes]
cat_idx = [list(cat_model_d5_2500.classes_).index(c) for c in classes]

hgb_test_proba = hgb_model.predict_proba(X_test_quality_dip)
et_test_proba = et_model.predict_proba(X_test_quality_dip)[:, et_idx]
cat_test_proba = cat_model_d5_2500.predict_proba(X_test_quality_dip)[:, cat_idx]

hgb_et_test_proba = 0.75 * hgb_test_proba + 0.25 * et_test_proba
final_proba = 0.60 * hgb_et_test_proba + 0.40 * cat_test_proba

final_pred = np.array(classes)[np.argmax(final_proba, axis=1)]

In [34]:
submission = pd.read_csv("../data/raw/sample_submission.csv")

submission["label"] = final_pred

assert len(submission) == 7000
assert submission["id"].equals(
    pd.read_csv("../data/raw/sample_submission.csv")["id"]
)
assert submission["label"].notna().all()
assert set(submission["label"]).issubset(set(classes))

output_path = "artifacts/submission_5_catboost_blend.csv"
submission.to_csv(output_path, index=False)

print("Saved:", output_path)
print("\nPrediction distribution:")
print(submission["label"].value_counts())
print("\nPreview:")
print(submission.head())

Saved: artifacts/submission_5_catboost_blend.csv

Prediction distribution:
label
instrument_glitch    2144
flare_star            985
pulsating_star        964
starspots             879
quiet_star            654
eclipsing_binary      544
background_binary     452
planet                378
Name: count, dtype: int64

Preview:
             id              label
0  TIC914436805             planet
1  TIC964940782         flare_star
2  TIC906185835         quiet_star
3  TIC927004741  instrument_glitch
4  TIC949385375  background_binary


In [35]:
import numpy as np
from sklearn.metrics import f1_score

classes = np.asarray(hgb_model.classes_)

# HGB + ExtraTrees baseline
hgb_et_val_proba = (
    0.75 * hgb_val_proba
    + 0.25 * et_val_proba
)

# Compare against the current 40% CatBoost blend
for cat_weight in [0.30, 0.40, 0.50]:
    blended_proba = (
        (1 - cat_weight) * hgb_et_val_proba
        + cat_weight * cat_val_proba
    )

    predictions = classes[np.argmax(blended_proba, axis=1)]

    score = f1_score(
        y_val,
        predictions,
        average="macro"
    )

    print(f"CatBoost weight: {cat_weight:.0%} | Macro-F1: {score:.4f}")

CatBoost weight: 30% | Macro-F1: 0.5354
CatBoost weight: 40% | Macro-F1: 0.5366
CatBoost weight: 50% | Macro-F1: 0.5352


In [36]:


classes = np.asarray(hgb_model.classes_)

# Your current best validation blend
hgb_et_val_proba = (
    0.75 * hgb_val_proba
    + 0.25 * et_val_proba
)

base_proba = (
    0.60 * hgb_et_val_proba
    + 0.40 * cat_val_proba
)

# Class frequencies from the training portion only
class_counts = pd.Series(y_train).value_counts()
class_priors = np.array([
    class_counts.get(c, 0) for c in classes
], dtype=float)

assert np.all(class_priors > 0), "A class is missing from training."

class_priors /= class_priors.sum()

baseline_pred = classes[np.argmax(base_proba, axis=1)]
baseline_f1 = f1_score(
    y_val, baseline_pred, average="macro"
)

print(f"Baseline macro-F1: {baseline_f1:.4f}")
print("\nTesting modest class-prior adjustments:")

results = []

for alpha in [0.00, 0.05, 0.10, 0.15, 0.20]:
    # alpha=0 means no adjustment
    adjusted_proba = base_proba / (
        class_priors ** alpha
    )

    adjusted_pred = classes[
        np.argmax(adjusted_proba, axis=1)
    ]

    score = f1_score(
        y_val, adjusted_pred, average="macro"
    )

    results.append((alpha, score))
    print(f"alpha={alpha:.2f} | Macro-F1={score:.4f}")

print("\nBest tested setting:", max(results, key=lambda x: x[1]))

Baseline macro-F1: 0.5366

Testing modest class-prior adjustments:
alpha=0.00 | Macro-F1=0.5366
alpha=0.05 | Macro-F1=0.5342
alpha=0.10 | Macro-F1=0.5325
alpha=0.15 | Macro-F1=0.5313
alpha=0.20 | Macro-F1=0.5306

Best tested setting: (0.0, 0.5365645268709381)
